<a href="https://anonymous.4open.science/r/qlens-/notebooks/notebooks/colab_phys_lens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PhysLens on Colab A100 — checkpoint-safe Week A runner

**Purpose:** Phase 5 Week A experiments — random-direction control, pre-registered SCAS sweep, PhysLens-OC 4-way contest — on Colab Pro A100 with Google Drive for checkpoint persistence.

**Runtime:** A100 (40GB) — `Runtime → Change runtime type → A100 GPU`.

**Checkpoint safety:** every intermediate result atomically written to `/content/drive/MyDrive/PhysLens/`. Disconnects do not lose progress.

**Total time budget:** ~90 min A100 ≈ 18 compute units (well under 100-unit cap).

---

## Fast-path data staging (pick ONE of three)

| Path | Local upload | Colab time | When to use |
|---|---|---|---|
| **A (recommended)** | ~18 MB zip to Drive (seconds) | ~3 min PhysBench download | You have the local cache |
| **B (zero upload)** | 0 bytes | ~3 min data + ~15 min cache regen on Colab (~3 units) | You cannot upload anything |
| **C (HF Hub)** | Upload once to a private HF dataset | ~2 min pull from HF | Repeated runs; one-time setup |

PhysBench is **public** → never upload it. Colab pulls it straight from HF Hub.

---

## Execution order

1. Mount Drive → 2. Clone repo → 3. Install deps → 4. **Download PhysBench (HF)** → 5. **Fetch cache (A / B / C)** → 6. Smoke test → 7. Gate 1 → 8. Gate 2 → 9. Gate 3 → 10. Aggregate verdict

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/PhysLens')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
for sub in ['cache', 'results', 'logs']:
    (DRIVE_ROOT / sub).mkdir(exist_ok=True)

print('Drive mounted at', DRIVE_ROOT)
print('Contents:', sorted(p.name for p in DRIVE_ROOT.iterdir()))
!nvidia-smi | head -10

Mounted at /content/drive
Drive mounted at /content/drive/MyDrive/PhysLens
Contents: ['cache', 'data', 'logs', 'results', 'upload_phys_lens_bundle.zip']
Sat Apr 18 18:04:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             42W /  400W |       0MiB /  40960MiB |      0%   

## 2. Clone / pull repo (ephemeral /content)

In [2]:
import os, subprocess
REPO_URL = 'https://anonymous.4open.science/r/qlens-.git'
BRANCH = 'physics-steering'
REPO_DIR = '/content/VLAs'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

%cd /content/VLAs
!git log --oneline -5

/content/VLAs
d54a9ac (HEAD -> physics-steering, origin/physics-steering) Fix bundle_for_colab: keep cache/ prefix in archive arcnames
5fd50bb Refactor PhysBench download script to use huggingface_hub for raw data retrieval and add skip-media option
0cfcc92 Refactor Colab PhysLens notebook for improved clarity and functionality
368c18d Add PhysLens predictor and SCAS random-direction control scripts
2550266 Fix PCA leakage + leakage-free SCAS validation at alpha=5


## 3. Install deps

In [3]:
!pip install -q --upgrade pip
!pip install -q 'transformers>=4.48' 'accelerate>=0.33' 'bitsandbytes>=0.44' \
    'peft>=0.13' 'qwen-vl-utils' 'safetensors' 'sentencepiece' 'protobuf' \
    'scikit-learn' 'scipy' 'h5py' 'Pillow' 'einops' 'timm' 'datasets' \
    'huggingface_hub'
!pip install -q -e /content/VLAs

import torch, transformers, bitsandbytes as bnb
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'avail', torch.cuda.is_available())
print('transformers', transformers.__version__)
print('bitsandbytes', bnb.__version__)

# HF token from Colab Secrets (needed for gated models, not Qwen3-VL).
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
    if tok:
        os.environ['HF_TOKEN'] = tok
        os.environ['HUGGING_FACE_HUB_TOKEN'] = tok
        print('HF_TOKEN loaded from Colab Secrets')
except Exception:
    print('No HF token (OK for Qwen3-VL — it is public)')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vla-physics-probing (pyproject.toml) ... done
torch 2.10.0+cu128 cuda 12.8 avail True
transformers 5.0.0
bitsandbytes 0.49.2
HF_TOKEN loaded from Colab Secrets


## 4. Download PhysBench DIRECTLY from HuggingFace (no local upload)

PhysBench is at `USC-GVL/PhysBench` on HF Hub. Colab → HF is ~100 MB/s, so this runs in 2–5 min instead of hours of Drive upload.

The repo's `scripts/download_physbench.py` handles merging GT answer files too.

In [4]:
%cd /content/VLAs
import pathlib

PB_DIR = pathlib.Path('/content/VLAs/data/physbench')
PB_DIR.mkdir(parents=True, exist_ok=True)

# Fast path: datasets library pulls the whole PhysBench dataset from HF Hub.
# Then scripts/download_physbench.py merges the answer JSONs.
if not (PB_DIR / 'val.json').exists():
    !python scripts/download_physbench.py --data-dir /content/VLAs/data/physbench
else:
    print('PhysBench already present at', PB_DIR)

# Verify.
!ls -la /content/VLAs/data/physbench/ | head -15

/content/VLAs
Target directory: /content/VLAs/data/physbench

--- Step 1: Download answer files ---
  val_answer.json: 200 entries
  test_answer.json: 10002 entries

--- Step 2: Download questions ---
Pulling raw PhysBench files from HF Hub (USC-GVL/PhysBench)...
Fetching 3 files: 100% 3/3 [00:28<00:00,  9.55s/it]
Download complete: : 7.43GB [00:28, 323MB/s]                Snapshot: /root/.cache/huggingface/hub/datasets--USC-GVL--PhysBench/snapshots/478fd93da8ec8d6f5252b9586b1fa10f335c5a95
  copied test.json
  extracting image.zip...
Download complete: : 7.43GB [00:28, 259MB/s]
  extracting video.zip...
  loaded test.json: 10002 questions

--- Step 3: Merge answers into questions and save ---
  Saved val.json: 200 questions, 199 with answers
  Saved test.json: 9802 questions, 9786 with answers
  Saved all.json: 10002 total questions

--- Validation ---
val.json: 200 questions
  Sample keys: ['scene', 'object', 'source', 'file_name', 'description', 'question', 'mode', 'idx', 'split', 'a

## 5. Fetch cache — pick ONE path (A, B, or C)

The cache is `cache/week1/features/qwen3-vl-8b_train/` — ~34 MB uncompressed, **18 MB zipped**.
Contains post-projection activations from 1793 training samples (disjoint from val).

### Run the first cell that matches your setup.

### Path A — Drive zip (recommended)

**Local prep (takes seconds):**
```bash
cd /content/qlens
python scripts/bundle_for_colab.py
# -> ./upload_phys_lens_bundle.zip  (~18 MB)
```

Drag-and-drop that zip into `MyDrive/PhysLens/` via the Drive web UI (18 MB uploads in seconds). Then run the cell below.

In [6]:
# Path A: unzip the bundle from Drive.
import pathlib, zipfile, time

bundle = pathlib.Path('/content/drive/MyDrive/PhysLens/upload_phys_lens_bundle.zip')
cache_dst = pathlib.Path('/content/VLAs/cache')

if bundle.exists():
    t0 = time.time()
    cache_dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle) as z:
        z.extractall(cache_dst.parent)  # bundle includes 'cache/week1/features/...'
    print(f'Extracted {bundle.name} in {time.time()-t0:.1f}s')
    !ls -la /content/VLAs/cache/week1/features/ | head -10
else:
    print('Path A not taken — bundle not found at', bundle)
    print('Either run this cell after uploading the bundle, OR use Path B or C below.')

Extracted upload_phys_lens_bundle.zip in 0.4s
total 12
drwxr-xr-x 3 root root 4096 Apr 18 18:06 .
drwxr-xr-x 3 root root 4096 Apr 18 18:06 ..
drwxr-xr-x 6 root root 4096 Apr 18 18:06 qwen3-vl-8b_train


### Path B — Zero upload (regenerate on Colab)

Run `scripts/extract_training_features.py` directly on Colab. ~15 min A100 time ≈ 3 compute units, no upload needed. Saves to Drive after for future runs.

In [ ]:
# Path B: regenerate cache on Colab.
# Only run this if Path A was not taken AND you don't want to upload anything.
import pathlib, subprocess

cache_qwen3 = pathlib.Path('/content/VLAs/cache/week1/features/qwen3-vl-8b_train')

if cache_qwen3.exists() and any(cache_qwen3.iterdir()):
    print('Cache already present at', cache_qwen3, '— skipping regen.')
else:
    print('Regenerating Qwen3-VL-8B training-split features on A100...')
    subprocess.run([
        'python', 'scripts/extract_training_features.py',
        '--model', 'qwen3-vl-8b',
        '--output-dir', 'cache/week1/features',
    ], cwd='/content/VLAs', check=True)

    # Back up to Drive so next session doesn't need to regenerate.
    import shutil
    drive_cache = pathlib.Path('/content/drive/MyDrive/PhysLens/cache/week1/features')
    drive_cache.mkdir(parents=True, exist_ok=True)
    dest = drive_cache / 'qwen3-vl-8b_train'
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(cache_qwen3, dest)
    print('Backed up cache to', dest)

### Path C — HuggingFace Hub relay

**Local prep (once):**
```bash
huggingface-cli login
huggingface-cli repo create physlens-cache --type dataset --private
huggingface-cli upload <your-username>/physlens-cache cache/week1/features/qwen3-vl-8b_train . --repo-type dataset
```

Then on Colab, pulls fast (~1 min), and subsequent Colab sessions reuse without re-upload.

In [18]:
# Path C: pull cache from a private HF dataset.
# Requires HF_TOKEN in Colab Secrets (step 3 loaded it if present).
HF_DATASET_REPO = '<your-username>/physlens-cache'   # <-- EDIT THIS LINE

if HF_DATASET_REPO.startswith('<'):
    print('Path C skipped — edit HF_DATASET_REPO above with your repo name.')
else:
    from huggingface_hub import snapshot_download
    import pathlib, shutil
    dst = pathlib.Path('/content/VLAs/cache/week1/features/qwen3-vl-8b_train')
    if dst.exists() and any(dst.iterdir()):
        print('Cache already present; skipping HF pull')
    else:
        local_dir = snapshot_download(
            repo_id=HF_DATASET_REPO,
            repo_type='dataset',
            local_dir='/content/hf_cache_dl',
        )
        dst.parent.mkdir(parents=True, exist_ok=True)
        # Move contents into the expected layout.
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(local_dir, dst)
        print('Pulled cache into', dst)

Path C skipped — edit HF_DATASET_REPO above with your repo name.


### Verify cache is in place (any path)

In [7]:
import pathlib
cache_dir = pathlib.Path('/content/VLAs/cache/week1/features/qwen3-vl-8b_train')
if cache_dir.exists():
    files = sorted(cache_dir.rglob('*'))
    sizes = sum(f.stat().st_size for f in files if f.is_file())
    print(f'Cache OK: {cache_dir}')
    print(f'  {len([f for f in files if f.is_file()])} files, {sizes/1e6:.1f} MB')
    print(f'  sites: {sorted(p.name for p in cache_dir.iterdir() if p.is_dir())}')
    # Route Colab outputs to Drive for persistence.
    import os
    pathlib.Path('/content/VLAs/results_drive').mkdir(exist_ok=True)
    pathlib.Path('/content/VLAs/logs_drive').mkdir(exist_ok=True)
    drive_res = pathlib.Path('/content/drive/MyDrive/PhysLens/results')
    drive_log = pathlib.Path('/content/drive/MyDrive/PhysLens/logs')
    if not pathlib.Path('/content/VLAs/results_drive').is_symlink():
        if pathlib.Path('/content/VLAs/results_drive').exists():
            import shutil; shutil.rmtree('/content/VLAs/results_drive')
        os.symlink(drive_res, '/content/VLAs/results_drive')
    if not pathlib.Path('/content/VLAs/logs_drive').is_symlink():
        if pathlib.Path('/content/VLAs/logs_drive').exists():
            import shutil; shutil.rmtree('/content/VLAs/logs_drive')
        os.symlink(drive_log, '/content/VLAs/logs_drive')
    print(f'  symlinks: results_drive -> {drive_res}')
    print(f'            logs_drive    -> {drive_log}')
else:
    print(f'CACHE MISSING at {cache_dir}')
    print('Go back and run Path A, B, or C.')

Cache OK: /content/VLAs/cache/week1/features/qwen3-vl-8b_train
  2001 files, 27.1 MB
  sites: ['enc_out', 'llm_16', 'llm_8', 'post_proj']
  symlinks: results_drive -> /content/drive/MyDrive/PhysLens/results
            logs_drive    -> /content/drive/MyDrive/PhysLens/logs


## 6. Smoke test (3 seeds, 20 samples, ~5 min)

In [20]:
%cd /content/VLAs
!python scripts/week3_scas_random_control.py \
    --model qwen3-vl-8b \
    --seeds 3 --max-samples 20 --alpha 5.0 \
    --cache-dir /content/VLAs/cache/week1 \
    --data-dir /content/VLAs/data/physbench \
    --output-dir /content/VLAs/results_drive/week4/random_control_smoke \
    --log-dir /content/VLAs/logs_drive \
    --include-baseline

/content/VLAs
2026-04-17 22:33:12 [INFO] week3_scas_random_control_qwen3-vl-8b: Logging to /content/VLAs/logs_drive/week3_scas_random_control_qwen3-vl-8b_20260417_223312.log
2026-04-17 22:33:12 [INFO] week3_scas_random_control_qwen3-vl-8b: ======================================================================
2026-04-17 22:33:12 [INFO] week3_scas_random_control_qwen3-vl-8b: SCAS random-direction control
2026-04-17 22:33:12 [INFO] week3_scas_random_control_qwen3-vl-8b:   model=qwen3-vl-8b  alpha=5.0  low_var_k=64
2026-04-17 22:33:12 [INFO] week3_scas_random_control_qwen3-vl-8b:   seeds=0..2  eval_split=val
2026-04-17 22:33:12 [INFO] week3_scas_random_control_qwen3-vl-8b:   output_dir=/content/VLAs/results_drive/week4/random_control_smoke
2026-04-17 22:33:12 [INFO] week3_scas_random_control_qwen3-vl-8b: ======================================================================
2026-04-17 22:33:12 [INFO] week3_scas_random_control_qwen3-vl-8b: Computing REAL steering vector (for PCA basis reus

## 7. Gate 1 — Random-direction control (20 seeds, val n=200, ~70 min)

In [21]:
%cd /content/VLAs
!python scripts/week3_scas_random_control.py \
    --model qwen3-vl-8b \
    --seeds 20 --alpha 5.0 \
    --cache-dir /content/VLAs/cache/week1 \
    --data-dir /content/VLAs/data/physbench \
    --output-dir /content/VLAs/results_drive/week4/random_control \
    --log-dir /content/VLAs/logs_drive \
    --include-baseline

/content/VLAs
2026-04-17 22:35:09 [INFO] week3_scas_random_control_qwen3-vl-8b: Logging to /content/VLAs/logs_drive/week3_scas_random_control_qwen3-vl-8b_20260417_223509.log
2026-04-17 22:35:09 [INFO] week3_scas_random_control_qwen3-vl-8b: ======================================================================
2026-04-17 22:35:09 [INFO] week3_scas_random_control_qwen3-vl-8b: SCAS random-direction control
2026-04-17 22:35:09 [INFO] week3_scas_random_control_qwen3-vl-8b:   model=qwen3-vl-8b  alpha=5.0  low_var_k=64
2026-04-17 22:35:09 [INFO] week3_scas_random_control_qwen3-vl-8b:   seeds=0..19  eval_split=val
2026-04-17 22:35:09 [INFO] week3_scas_random_control_qwen3-vl-8b:   output_dir=/content/VLAs/results_drive/week4/random_control
2026-04-17 22:35:09 [INFO] week3_scas_random_control_qwen3-vl-8b: ======================================================================
2026-04-17 22:35:09 [INFO] week3_scas_random_control_qwen3-vl-8b: Computing REAL steering vector (for PCA basis reuse)...

In [22]:
# Gate 1 verdict.
import json, pathlib
p = pathlib.Path('/content/VLAs/results_drive/week4/random_control/qwen3-vl-8b/random_control_summary.json')
if p.exists():
    s = json.loads(p.read_text())
    for k in ['n_seeds','baseline_acc_quant','random_acc_quant_median',
              'random_delta_quant_median','random_delta_quant_ci95',
              'observed_scas_delta_quant','kill_threshold_delta_quant',
              'kill_gate_fired','verdict']:
        print(f'{k:34s} = {s.get(k)}')
else:
    print('No summary yet — finish Gate 1 first.')

n_seeds                            = 20
baseline_acc_quant                 = 0.7272727272727273
random_acc_quant_median            = 0.7272727272727273
random_delta_quant_median          = 0.0
random_delta_quant_ci95            = [-0.005454545454545457, 0.004545454545454547]
observed_scas_delta_quant          = 0.0364
kill_threshold_delta_quant         = 0.0182
kill_gate_fired                    = False
verdict                            = SCAS median delta EXCEEDS random controls — headline survives this gate


## 8. Gate 2 — Pre-registered α={3, 5} sweep

In [23]:
%cd /content/VLAs
!python scripts/week3_scas_sweep.py \
    --model qwen3-vl-8b \
    --alphas 0 3 5 --method amplify \
    --pca-split train --eval-split val \
    --cache-dir /content/VLAs/cache/week1 \
    --data-dir /content/VLAs/data/physbench \
    --output-dir /content/VLAs/results_drive/week4/scas_prereg \
    --log-dir /content/VLAs/logs_drive

/content/VLAs
2026-04-17 23:10:02 [INFO] week3_scas_qwen3-vl-8b: Logging to /content/VLAs/logs_drive/week3_scas_qwen3-vl-8b_20260417_231002.log
2026-04-17 23:10:02 [INFO] week3_scas_qwen3-vl-8b: ======================================================================
2026-04-17 23:10:02 [INFO] week3_scas_qwen3-vl-8b: SCAS alpha-sweep: model=qwen3-vl-8b, alphas=[0.0, 3.0, 5.0]
2026-04-17 23:10:02 [INFO] week3_scas_qwen3-vl-8b:   method=amplify, low_var_k=64
2026-04-17 23:10:02 [INFO] week3_scas_qwen3-vl-8b:   pca_split=train, eval_split=val
2026-04-17 23:10:02 [INFO] week3_scas_qwen3-vl-8b: ======================================================================
2026-04-17 23:10:02 [INFO] week3_scas_qwen3-vl-8b: PCA source: 500 samples from 'train' split
2026-04-17 23:10:02 [INFO] week3_scas_qwen3-vl-8b: Computing steering vector from PCA-split features...
Loaded 9802 questions from data/physbench/test.json
2026-04-17 23:10:04 [INFO] week3_scas_qwen3-vl-8b:   method=amplify, K=64, dim=4096,

## 9. Gate 3 — PhysLens-OC 4-way contest

Contrast (PhysLens-OC novel variant) vs amplify vs qual-contrast positive control. Random is already on disk from Gate 1.

In [10]:
%cd /content/VLAs
# Contrast (PhysLens-OC).
!python scripts/week3_scas_sweep.py \
    --model qwen3-vl-8b --alphas 0 5 --method contrast \
    --pca-split train --eval-split val \
    --cache-dir /content/VLAs/cache/week1 \
    --data-dir /content/VLAs/data/physbench \
    --output-dir /content/VLAs/results_drive/week4/oc_contest/contrast \
    --log-dir /content/VLAs/logs_drive

/content/VLAs
2026-04-18 18:41:18 [INFO] week3_scas_qwen3-vl-8b: Logging to /content/VLAs/logs_drive/week3_scas_qwen3-vl-8b_20260418_184118.log
2026-04-18 18:41:18 [INFO] week3_scas_qwen3-vl-8b: ======================================================================
2026-04-18 18:41:18 [INFO] week3_scas_qwen3-vl-8b: SCAS alpha-sweep: model=qwen3-vl-8b, alphas=[0.0, 5.0]
2026-04-18 18:41:18 [INFO] week3_scas_qwen3-vl-8b:   method=contrast, low_var_k=64
2026-04-18 18:41:18 [INFO] week3_scas_qwen3-vl-8b:   pca_split=train, eval_split=val
2026-04-18 18:41:18 [INFO] week3_scas_qwen3-vl-8b: ======================================================================
2026-04-18 18:41:18 [INFO] week3_scas_qwen3-vl-8b: PCA source: 500 samples from 'train' split
2026-04-18 18:41:18 [INFO] week3_scas_qwen3-vl-8b: Computing steering vector from PCA-split features...
Loaded 9802 questions from data/physbench/test.json
2026-04-18 18:41:20 [INFO] week3_scas_qwen3-vl-8b:   method=contrast, K=64, dim=4096, V_

In [11]:
# Qual-contrast positive control: flip sign on contrast direction.
!python scripts/week3_scas_sweep.py \
    --model qwen3-vl-8b --alphas 0 -5 --method contrast \
    --pca-split train --eval-split val \
    --cache-dir /content/VLAs/cache/week1 \
    --data-dir /content/VLAs/data/physbench \
    --output-dir /content/VLAs/results_drive/week4/oc_contest/qual_contrast \
    --log-dir /content/VLAs/logs_drive

2026-04-18 18:45:54 [INFO] week3_scas_qwen3-vl-8b: Logging to /content/VLAs/logs_drive/week3_scas_qwen3-vl-8b_20260418_184554.log
2026-04-18 18:45:54 [INFO] week3_scas_qwen3-vl-8b: ======================================================================
2026-04-18 18:45:54 [INFO] week3_scas_qwen3-vl-8b: SCAS alpha-sweep: model=qwen3-vl-8b, alphas=[0.0, -5.0]
2026-04-18 18:45:54 [INFO] week3_scas_qwen3-vl-8b:   method=contrast, low_var_k=64
2026-04-18 18:45:54 [INFO] week3_scas_qwen3-vl-8b:   pca_split=train, eval_split=val
2026-04-18 18:45:54 [INFO] week3_scas_qwen3-vl-8b: ======================================================================
2026-04-18 18:45:54 [INFO] week3_scas_qwen3-vl-8b: PCA source: 500 samples from 'train' split
2026-04-18 18:45:54 [INFO] week3_scas_qwen3-vl-8b: Computing steering vector from PCA-split features...
Loaded 9802 questions from data/physbench/test.json
2026-04-18 18:45:56 [INFO] week3_scas_qwen3-vl-8b:   method=contrast, K=64, dim=4096, V_low explains 

In [8]:
import os, pathlib, shutil
for src, dst in [
    ('/content/drive/MyDrive/PhysLens/results', '/content/VLAs/results_drive'),
    ('/content/drive/MyDrive/PhysLens/logs',    '/content/VLAs/logs_drive'),
]:
    if os.path.lexists(dst):
        (os.unlink if os.path.islink(dst) else shutil.rmtree)(dst)
    os.symlink(src, dst)
print('symlinks set')

symlinks set


In [9]:
import pathlib, re, json
log_dir = pathlib.Path('/content/drive/MyDrive/PhysLens/logs')
logs = sorted(log_dir.glob('week3_scas_qwen3-vl-8b_*.log'), key=lambda p: p.stat().st_mtime)
# Find the log file that ran eval_split=test
test_logs = [l for l in logs if 'eval_split=test' in l.read_text()]
if not test_logs:
    print('No test-set log found');
else:
    text = test_logs[-1].read_text()
    print('Parsing:', test_logs[-1].name)
    pat = re.compile(r'alpha=([\-\d\.]+):\s+acc_quant=([\d\.]+)\s+acc_qual=([\d\.]+)\s+acc_all=([\d\.]+)')
    sweep = [{'alpha': float(m[1]), 'acc_quant': float(m[2]),
              'acc_qual': float(m[3]), 'acc_all': float(m[4])} for m in pat.finditer(text)]
    print('Recovered:', sweep)
    out = pathlib.Path('/content/drive/MyDrive/PhysLens/results/week4/scas_testset_amplify_recovered.json')
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps({
        'model': 'qwen3-vl-8b', 'method': 'amplify', 'split': 'test',
        'note': 'Recovered from log after runtime disconnect',
        'sweep': sweep}, indent=2))
    print('Wrote', out)

Parsing: week3_scas_qwen3-vl-8b_20260417_234725.log
Recovered: [{'alpha': 0.0, 'acc_quant': 0.7187, 'acc_qual': 0.4653, 'acc_all': 0.4911}, {'alpha': 3.0, 'acc_quant': 0.7107, 'acc_qual': 0.467, 'acc_all': 0.4918}, {'alpha': 5.0, 'acc_quant': 0.6997, 'acc_qual': 0.4663, 'acc_all': 0.4901}]
Wrote /content/drive/MyDrive/PhysLens/results/week4/scas_testset_amplify_recovered.json


In [12]:
# After Gate 3: test-set run (will take ~90 min on A100)
!python scripts/week3_scas_sweep.py \
    --model qwen3-vl-8b --alphas 0 3 5 --method amplify \
    --pca-split train --eval-split test \
    --cache-dir /content/VLAs/cache/week1 \
    --data-dir /content/VLAs/data/physbench \
    --output-dir /content/VLAs/results_drive/week4/scas_testset_amplify \
    --log-dir /content/VLAs/logs_drive

!python scripts/week3_scas_sweep.py \
    --model qwen3-vl-8b --alphas 0 5 --method contrast \
    --pca-split train --eval-split test \
    --cache-dir /content/VLAs/cache/week1 \
    --data-dir /content/VLAs/data/physbench \
    --output-dir /content/VLAs/results_drive/week4/scas_testset_contrast \
    --log-dir /content/VLAs/logs_drive

2026-04-18 18:49:35 [INFO] week3_scas_qwen3-vl-8b: Logging to /content/VLAs/logs_drive/week3_scas_qwen3-vl-8b_20260418_184935.log
2026-04-18 18:49:35 [INFO] week3_scas_qwen3-vl-8b: ======================================================================
2026-04-18 18:49:35 [INFO] week3_scas_qwen3-vl-8b: SCAS alpha-sweep: model=qwen3-vl-8b, alphas=[0.0, 3.0, 5.0]
2026-04-18 18:49:35 [INFO] week3_scas_qwen3-vl-8b:   method=amplify, low_var_k=64
2026-04-18 18:49:35 [INFO] week3_scas_qwen3-vl-8b:   pca_split=train, eval_split=test
2026-04-18 18:49:35 [INFO] week3_scas_qwen3-vl-8b: ======================================================================
2026-04-18 18:49:35 [INFO] week3_scas_qwen3-vl-8b: PCA source: 500 samples from 'train' split
2026-04-18 18:49:35 [INFO] week3_scas_qwen3-vl-8b: Computing steering vector from PCA-split features...
Loaded 9802 questions from data/physbench/test.json
2026-04-18 18:49:37 [INFO] week3_scas_qwen3-vl-8b:   method=amplify, K=64, dim=4096, V_low explai

## 10. Final aggregation

In [13]:
import json, pathlib, os as _os
def load(p):
    p = pathlib.Path(p)
    return json.loads(p.read_text()) if p.exists() else None
base = '/content/VLAs/results_drive/week4'
rc     = load(f'{base}/random_control/qwen3-vl-8b/random_control_summary.json')
prereg = load(f'{base}/scas_prereg/scas_sweep_qwen3-vl-8b.json')
con    = load(f'{base}/oc_contest/contrast/scas_sweep_qwen3-vl-8b.json')
qual   = load(f'{base}/oc_contest/qual_contrast/scas_sweep_qwen3-vl-8b.json')

def delta(s, a):
    if s is None: return (None,None)
    b = next((x for x in s['sweep'] if x['alpha']==0.0), None)
    t = next((x for x in s['sweep'] if abs(x['alpha']-a)<1e-6), None)
    if b is None or t is None: return (None,None)
    return (t['acc_quant']-b['acc_quant'], t['acc_qual']-b['acc_qual'])

print('='*78)
print(' PHYSLENS WEEK A — HEADLINE TABLE (Qwen3-VL-8B, PhysBench val n=200)')
print('='*78)
print(f'  {"condition":<32} {"alpha":>8} {"d_quant":>10} {"d_qual":>10}')
print(f'  {"-"*62}')
if prereg is not None:
    for a in [3.0, 5.0]:
        dq, dl = delta(prereg, a)
        if dq is not None:
            print(f'  {"SCAS amplify (pre-reg)":<32} {a:>8.1f} {dq:+.4f} {dl:+.4f}')
if con is not None:
    dq, dl = delta(con, 5.0)
    if dq is not None: print(f'  {"PhysLens-OC contrast":<32} {5.0:>8.1f} {dq:+.4f} {dl:+.4f}')
if qual is not None:
    dq, dl = delta(qual, -5.0)
    if dq is not None: print(f'  {"qual-contrast (pos. control)":<32} {-5.0:>8.1f} {dq:+.4f} {dl:+.4f}')
if rc is not None:
    print(f'  {"random-direction (median)":<32} {5.0:>8.1f} {rc["random_delta_quant_median"]:+.4f} {rc["random_delta_qual_median"]:+.4f}')
    ci = rc['random_delta_quant_ci95']
    print(f'    random 95% CI on d_quant: [{ci[0]:+.4f}, {ci[1]:+.4f}]  (n_seeds={rc["n_seeds"]})')
print('='*78)

# Gate verdict.
print('\nGATES:')
if rc is not None:
    print(f'  Gate 1 (random): kill_fired={rc["kill_gate_fired"]}')
    print(f'    {rc["verdict"]}')
else:
    print('  Gate 1: not yet run')

# Save verdict.
verdict = {'gate_1_random_control': rc, 'prereg_sweep': prereg,
           'oc_contrast': con, 'oc_qual_contrast': qual}
vp = pathlib.Path(f'{base}/week_a_verdict.json')
tmp = vp.with_suffix('.tmp')
with open(tmp, 'w') as f:
    json.dump(verdict, f, indent=2, default=str)
    f.flush(); _os.fsync(f.fileno())
_os.replace(tmp, vp)
print(f'\nWrote verdict: {vp}')

 PHYSLENS WEEK A — HEADLINE TABLE (Qwen3-VL-8B, PhysBench val n=200)
  condition                           alpha    d_quant     d_qual
  --------------------------------------------------------------
  SCAS amplify (pre-reg)                3.0 +0.0182 +0.0000
  SCAS amplify (pre-reg)                5.0 +0.0182 -0.0069
  PhysLens-OC contrast                  5.0 +0.0364 +0.0069
  qual-contrast (pos. control)         -5.0 +0.0000 -0.0069
  random-direction (median)             5.0 +0.0000 -0.0069
    random 95% CI on d_quant: [-0.0055, +0.0045]  (n_seeds=20)

GATES:
  Gate 1 (random): kill_fired=False
    SCAS median delta EXCEEDS random controls — headline survives this gate

Wrote verdict: /content/VLAs/results_drive/week4/week_a_verdict.json


## 11. Keep-alive (optional; only during active runs)

In [14]:
# from IPython.display import Javascript, display
# display(Javascript('setInterval(()=>document.querySelector("colab-toolbar-button#connect").click(),60000)'))
print('(keep-alive disabled — uncomment only during active A100 runs)')

(keep-alive disabled — uncomment only during active A100 runs)
